In [1]:
import json
import os, statistics, math

def extract_step_to_acc(path: str):
    """
    Read a raw metrics file and return {step: test_acc} from even-numbered lines only.
    - 1-based line numbering (keep only even lines)
    - JSON parse; keep only split == 'test'
    - Map: step -> acc (latest occurrence wins if duplicated)
    """
    result = {}
    with open(path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f, start=1):  # 1-based
            line = line.strip()
            if not line or (i % 2 != 0):  # skip odd lines
                continue
            try:
                obj = json.loads(line)
            except json.JSONDecodeError:
                continue
            if obj.get("split") != "test":
                continue
            step = obj.get("step")
            acc = obj.get("acc")
            if step is not None and acc is not None:
                result[step] = acc  # latest one wins
    return result

In [2]:
def summarize_by_step(root='.', start=1, end=53, prefix='client_', ext='.raw',
                      variance='population', step_keys=None, require_all=False):
    """
    Aggregate across clients PER STEP and return TWO dictionaries:
      - mean_by_step: {step: mean_test_acc_over_clients}
      - var_by_step:  {step: variance_test_acc_over_clients}

    Parameters:
      - variance: 'population' (statistics.pvariance) | 'sample' (statistics.variance)
      - step_keys: optional iterable of steps to enforce (e.g., [0,10,20,...,100]).
                   If None, uses the union of steps observed across clients.
      - require_all: if True, only include a step if ALL clients have a value for it.
                     If False, compute from available values.

    Notes:
      - Uses extract_step_to_acc(path) from earlier cell (even-line, split=='test').
      - Missing files or missing steps are ignored per 'require_all' policy.
    """
    # Collect per-step lists of accuracies across clients
    accs_by_step = {}
    client_count = 0
    for i in range(start, end + 1):
        path = os.path.join(root, f"{prefix}{i:03d}{ext}")
        try:
            d = extract_step_to_acc(path)
        except FileNotFoundError:
            # Missing client file; skip
            continue
        client_count += 1
        for step, acc in d.items():
            accs_by_step.setdefault(step, []).append(acc)

    # Decide which steps to include
    if step_keys is None:
        steps = sorted(accs_by_step.keys())
    else:
        steps = list(step_keys)

    mean_by_step = {}
    var_by_step  = {}
    for step in steps:
        vals = accs_by_step.get(step, [])
        if not vals:
            continue  # no data at this step
        if require_all and len(vals) != client_count:
            # Skip this step because not all clients provided it
            continue
        m = sum(vals) / len(vals)
        if variance == 'population':
            v = statistics.pvariance(vals)
        elif variance == 'sample':
            v = statistics.variance(vals) if len(vals) > 1 else float('nan')
        else:
            raise ValueError("variance must be 'population' or 'sample'")
        mean_by_step[step] = m
        var_by_step[step]  = v
    return mean_by_step, var_by_step

In [3]:
mean_dict, var_dict = summarize_by_step(root='.', start=1, end=53)
print(mean_dict)
print(var_dict)

{0: 0.787, 10: 0.7775000000000001, 20: 0.7825, 30: 0.7970000000000002, 40: 0.7945000000000001, 50: 0.8009999999999999, 60: 0.8099999999999999, 70: 0.8110000000000002, 80: 0.806, 90: 0.806, 100: 0.8089999999999998, 110: 0.8145, 120: 0.8079999999999998, 130: 0.8154999999999999, 140: 0.8180000000000002, 150: 0.8164999999999999, 160: 0.8145000000000001, 170: 0.8280000000000001, 180: 0.8310000000000001, 190: 0.8249999999999998, 200: 0.8290000000000001, 210: 0.834, 220: 0.8300000000000003, 230: 0.8274999999999999, 240: 0.8300000000000001, 250: 0.8365, 260: 0.8399999999999996, 270: 0.8345945945945946, 280: 0.8400000000000001, 290: 0.8438888888888889, 300: 0.8433333333333334}
{0: 0.013631, 10: 0.013103749999999999, 20: 0.010563749999999998, 30: 0.009411, 40: 0.00951975, 50: 0.009879, 60: 0.007200000000000001, 70: 0.008919000000000002, 80: 0.008663999999999998, 90: 0.010043999999999999, 100: 0.007519, 110: 0.0075397500000000004, 120: 0.006676, 130: 0.0069897499999999994, 140: 0.006536, 150: 0.0